# UEBA Results Analysis Notebook

This notebook analyzes the outputs generated by the UEBA pipeline.

It combines several analysis formats:

- data previews;
- summary tables;
- descriptive statistics;
- visual charts;
- short analytical interpretations.

Input files:

- `data/processed/ueba_features.csv`
- `data/alerts/alerts.csv`

The goal is to support the analytical report with clear statistics and visual evidence.

## 1. Imports and project paths

In [ ]:
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt


PROJECT_ROOT = Path("..").resolve()

FEATURES_FILE = PROJECT_ROOT / "data" / "processed" / "ueba_features.csv"
ALERTS_FILE = PROJECT_ROOT / "data" / "alerts" / "alerts.csv"
FIGURES_DIR = PROJECT_ROOT / "reports" / "figures"

FIGURES_DIR.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)
print("Features file:", FEATURES_FILE)
print("Alerts file:", ALERTS_FILE)
print("Figures directory:", FIGURES_DIR)

## 2. Load pipeline outputs

This section loads the final UEBA outputs generated by the Python pipeline.

- `ueba_features.csv` contains all analyzed user/day behaviors.
- `alerts.csv` contains only behaviors where `risk_score > 0`.

In [ ]:
features_df = pd.read_csv(FEATURES_FILE)
alerts_df = pd.read_csv(ALERTS_FILE)

print("Features shape:", features_df.shape)
print("Alerts shape:", alerts_df.shape)

### 2.1 Preview of processed UEBA features

In [ ]:
display(features_df.head())

### 2.2 Preview of generated alerts

In [ ]:
display(alerts_df.head())

## 3. Global project statistics

This table gives a first overview of the pipeline output.

In [ ]:
total_behaviors = len(features_df)
total_alerts = len(alerts_df)
alert_rate = total_alerts / total_behaviors * 100 if total_behaviors > 0 else 0

critical_alerts = (
    (features_df["risk_level"] == "critical").sum()
    if "risk_level" in features_df.columns
    else None
)

global_stats = pd.DataFrame(
    {
        "Metric": [
            "Total analyzed behaviors",
            "Total alerts",
            "Alert rate (%)",
            "Unique users in features",
            "Unique users in alerts",
            "Critical alerts",
            "Number of columns in final dataset",
        ],
        "Value": [
            total_behaviors,
            total_alerts,
            round(alert_rate, 2),
            features_df["user"].nunique() if "user" in features_df.columns else None,
            alerts_df["user"].nunique() if "user" in alerts_df.columns else None,
            critical_alerts,
            features_df.shape[1],
        ],
    }
)

display(global_stats)

### Interpretation

The alert rate shows the proportion of user/day behaviors that generated a positive risk score.

A UEBA system should normally produce fewer alerts than total behaviors, because most user activity is expected to be normal or low-risk.

## 4. Risk score descriptive statistics

In [ ]:
risk_score_stats = alerts_df["risk_score"].describe().reset_index()
risk_score_stats.columns = ["Statistic", "Risk score"]

display(risk_score_stats)

### Interpretation

The descriptive statistics help understand how alert severity is distributed.

- The minimum shows the lowest alert score.
- The maximum should not exceed 100.
- The mean and quartiles help identify whether alerts are mostly low-risk or concentrated around high-risk values.

## 5. Risk level distribution

In [ ]:
risk_distribution = (
    features_df["risk_level"]
    .value_counts()
    .rename_axis("risk_level")
    .reset_index(name="count")
)

risk_distribution["percentage"] = (
    risk_distribution["count"] / risk_distribution["count"].sum() * 100
).round(2)

display(risk_distribution)

In [ ]:
plt.figure(figsize=(8, 5))
plt.bar(risk_distribution["risk_level"], risk_distribution["count"])
plt.title("Risk Level Distribution")
plt.xlabel("Risk level")
plt.ylabel("Count")
plt.tight_layout()

risk_fig_path = FIGURES_DIR / "risk_level_distribution.png"
plt.savefig(risk_fig_path, dpi=150)
plt.show()

print("Saved figure:", risk_fig_path)

### Interpretation

The risk level distribution provides a high-level view of the alert landscape.

A large number of low-risk behaviors is expected. Critical alerts should remain a minority, but they are the most important cases for investigation.

## 6. Top risky users

In [ ]:
top_users = (
    alerts_df.groupby("user")
    .agg(
        alert_count=("risk_score", "count"),
        max_risk_score=("risk_score", "max"),
        avg_risk_score=("risk_score", "mean"),
    )
    .sort_values(["alert_count", "max_risk_score"], ascending=False)
    .head(10)
    .reset_index()
)

top_users["avg_risk_score"] = top_users["avg_risk_score"].round(2)

display(top_users)

In [ ]:
plt.figure(figsize=(10, 5))
plt.bar(top_users["user"], top_users["alert_count"])
plt.title("Top 10 Risky Users by Alert Count")
plt.xlabel("User")
plt.ylabel("Alert count")
plt.xticks(rotation=45)
plt.tight_layout()

top_users_fig_path = FIGURES_DIR / "top_risky_users.png"
plt.savefig(top_users_fig_path, dpi=150)
plt.show()

print("Saved figure:", top_users_fig_path)

### Interpretation

This table and chart identify users who repeatedly generate alerts.

A high alert count does not automatically prove malicious behavior, but it highlights users that should be prioritized for review.

## 7. Isolation Forest anomaly analysis

In [ ]:
if "is_anomaly" in features_df.columns:
    isolation_summary = (
        features_df["is_anomaly"]
        .value_counts()
        .rename_axis("is_anomaly")
        .reset_index(name="count")
    )

    isolation_summary["label"] = isolation_summary["is_anomaly"].map(
        {0: "Normal", 1: "Anomaly"}
    )

    isolation_summary["percentage"] = (
        isolation_summary["count"] / isolation_summary["count"].sum() * 100
    ).round(2)

    display(isolation_summary)

    isolation_anomalies = int((features_df["is_anomaly"] == 1).sum())
    print("Isolation Forest anomalies:", isolation_anomalies)
else:
    print("Column is_anomaly not found.")

In [ ]:
if "anomaly_score" in features_df.columns:
    anomaly_stats = features_df["anomaly_score"].describe().reset_index()
    anomaly_stats.columns = ["Statistic", "Anomaly score"]
    display(anomaly_stats)
else:
    print("Column anomaly_score not found.")

### Interpretation

Isolation Forest detects rare behavioral patterns.

The anomaly score distribution gives insight into how strongly some behaviors differ from the learned baseline.

## 8. TensorFlow Autoencoder anomaly analysis

In [ ]:
if "autoencoder_is_anomaly" in features_df.columns:
    autoencoder_summary = (
        features_df["autoencoder_is_anomaly"]
        .value_counts()
        .rename_axis("autoencoder_is_anomaly")
        .reset_index(name="count")
    )

    autoencoder_summary["label"] = autoencoder_summary["autoencoder_is_anomaly"].map(
        {0: "Normal", 1: "Anomaly"}
    )

    autoencoder_summary["percentage"] = (
        autoencoder_summary["count"] / autoencoder_summary["count"].sum() * 100
    ).round(2)

    display(autoencoder_summary)

    autoencoder_anomalies = int((features_df["autoencoder_is_anomaly"] == 1).sum())
    print("TensorFlow Autoencoder anomalies:", autoencoder_anomalies)
else:
    print("Autoencoder columns not found. Run the pipeline with --use-autoencoder first.")

In [ ]:
if "autoencoder_reconstruction_error" in features_df.columns:
    reconstruction_stats = (
        features_df["autoencoder_reconstruction_error"]
        .describe()
        .reset_index()
    )

    reconstruction_stats.columns = ["Statistic", "Reconstruction error"]
    display(reconstruction_stats)

    plt.figure(figsize=(8, 5))
    plt.hist(features_df["autoencoder_reconstruction_error"], bins=30)
    plt.title("Autoencoder Reconstruction Error Distribution")
    plt.xlabel("Reconstruction error")
    plt.ylabel("Frequency")
    plt.tight_layout()

    reconstruction_fig_path = FIGURES_DIR / "autoencoder_reconstruction_error_distribution.png"
    plt.savefig(reconstruction_fig_path, dpi=150)
    plt.show()

    print("Saved figure:", reconstruction_fig_path)
else:
    print("Column autoencoder_reconstruction_error not found.")

In [ ]:
if "autoencoder_is_anomaly" in features_df.columns:
    plt.figure(figsize=(6, 4))
    plt.bar(
        autoencoder_summary["label"],
        autoencoder_summary["count"],
    )
    plt.title("TensorFlow Autoencoder Anomaly Distribution")
    plt.xlabel("Autoencoder classification")
    plt.ylabel("Count")
    plt.tight_layout()

    autoencoder_fig_path = FIGURES_DIR / "autoencoder_anomalies.png"
    plt.savefig(autoencoder_fig_path, dpi=150)
    plt.show()

    print("Saved figure:", autoencoder_fig_path)

### Interpretation

The Autoencoder identifies behaviors that are difficult to reconstruct from the learned behavioral representation.

High reconstruction errors are interpreted as stronger deviations from normal behavior.

## 9. Alerts by department

In [ ]:
if "department" in alerts_df.columns:
    alerts_by_department = (
        alerts_df.groupby("department")
        .agg(
            alert_count=("risk_score", "count"),
            max_risk_score=("risk_score", "max"),
            avg_risk_score=("risk_score", "mean"),
        )
        .sort_values("alert_count", ascending=False)
        .head(10)
        .reset_index()
    )

    alerts_by_department["avg_risk_score"] = alerts_by_department["avg_risk_score"].round(2)

    display(alerts_by_department)
else:
    print("Column department not found. Run the pipeline with --include-ldap first.")

In [ ]:
if "department" in alerts_df.columns:
    plt.figure(figsize=(12, 5))
    plt.bar(
        alerts_by_department["department"].astype(str),
        alerts_by_department["alert_count"],
    )
    plt.title("Top Departments by Alert Count")
    plt.xlabel("Department")
    plt.ylabel("Alert count")
    plt.xticks(rotation=45, ha="right")
    plt.tight_layout()

    department_fig_path = FIGURES_DIR / "alerts_by_department.png"
    plt.savefig(department_fig_path, dpi=150)
    plt.show()

    print("Saved figure:", department_fig_path)

### Interpretation

LDAP enrichment makes it possible to analyze alerts at the organizational level.

Departments with many alerts are not necessarily malicious departments, but they deserve deeper contextual review.

## 10. Alerts by supervisor

In [ ]:
if "supervisor" in alerts_df.columns:
    alerts_by_supervisor = (
        alerts_df.groupby("supervisor")
        .agg(
            alert_count=("risk_score", "count"),
            max_risk_score=("risk_score", "max"),
            avg_risk_score=("risk_score", "mean"),
        )
        .sort_values("alert_count", ascending=False)
        .head(10)
        .reset_index()
    )

    alerts_by_supervisor["avg_risk_score"] = alerts_by_supervisor["avg_risk_score"].round(2)

    display(alerts_by_supervisor)
else:
    print("Column supervisor not found. Run the pipeline with --include-ldap first.")

In [ ]:
if "supervisor" in alerts_df.columns:
    plt.figure(figsize=(12, 5))
    plt.bar(
        alerts_by_supervisor["supervisor"].astype(str),
        alerts_by_supervisor["alert_count"],
    )
    plt.title("Top Supervisors by Alert Count")
    plt.xlabel("Supervisor")
    plt.ylabel("Alert count")
    plt.xticks(rotation=45, ha="right")
    plt.tight_layout()

    supervisor_fig_path = FIGURES_DIR / "alerts_by_supervisor.png"
    plt.savefig(supervisor_fig_path, dpi=150)
    plt.show()

    print("Saved figure:", supervisor_fig_path)

### Interpretation

Supervisor-level analysis helps structure investigations.

It allows analysts to understand whether alert concentration is linked to specific teams or reporting lines.

## 11. Email and HTTP activity in alerts

In [ ]:
email_alerts = None
http_alerts = None

if "email_events" in alerts_df.columns:
    email_alerts = int((alerts_df["email_events"] > 0).sum())
    print("Alerts with email activity:", email_alerts)
else:
    print("Column email_events not found.")

if "http_events" in alerts_df.columns:
    http_alerts = int((alerts_df["http_events"] > 0).sum())
    print("Alerts with HTTP activity:", http_alerts)
else:
    print("Column http_events not found.")

activity_data = []

if email_alerts is not None:
    activity_data.append({"activity": "Email", "alert_count": email_alerts})

if http_alerts is not None:
    activity_data.append({"activity": "HTTP", "alert_count": http_alerts})

if activity_data:
    activity_df = pd.DataFrame(activity_data)
    display(activity_df)
else:
    activity_df = pd.DataFrame()

In [ ]:
if not activity_df.empty:
    plt.figure(figsize=(6, 4))
    plt.bar(activity_df["activity"], activity_df["alert_count"])
    plt.title("Alerts with Email or HTTP Activity")
    plt.xlabel("Activity type")
    plt.ylabel("Alert count")
    plt.tight_layout()

    activity_fig_path = FIGURES_DIR / "alerts_email_http_activity.png"
    plt.savefig(activity_fig_path, dpi=150)
    plt.show()

    print("Saved figure:", activity_fig_path)

### Interpretation

Email and HTTP activity provide additional behavioral context.

An alert involving email activity or web activity may indicate possible data leakage, suspicious browsing, or unusual communication behavior.

## 12. Correlation between risk indicators

In [ ]:
numeric_columns = [
    column
    for column in [
        "rule_score",
        "risk_score",
        "anomaly_score",
        "autoencoder_reconstruction_error",
        "email_events",
        "http_events",
        "file_copy_events",
        "usb_events",
    ]
    if column in features_df.columns
]

if numeric_columns:
    correlation_matrix = features_df[numeric_columns].corr().round(3)
    display(correlation_matrix)
else:
    print("No selected numeric columns found for correlation analysis.")

### Interpretation

The correlation table helps identify relationships between risk indicators.

For example, a relationship between `risk_score` and `rule_score` is expected because the final score is partly based on the rule engine. Other correlations can highlight behavioral patterns worth investigating.

## 13. High-risk alerts sample

In [ ]:
important_columns = [
    "user",
    "day",
    "risk_score",
    "risk_level",
    "alert_reason",
    "rule_score",
    "is_anomaly",
    "autoencoder_is_anomaly",
    "department",
    "team",
    "supervisor",
    "email_events",
    "http_events",
]

existing_columns = [
    column for column in important_columns
    if column in alerts_df.columns
]

high_risk_alerts = (
    alerts_df[existing_columns]
    .sort_values("risk_score", ascending=False)
    .head(15)
)

display(high_risk_alerts)

### Interpretation

This sample table is useful for investigation.

It combines technical signals, risk scoring, anomaly detection, and LDAP context in a single view.

## 14. Final summary table for the report

In [ ]:
summary = {
    "total_behaviors": total_behaviors,
    "total_alerts": total_alerts,
    "alert_rate_percent": round(alert_rate, 2),
    "critical_alerts": int((features_df["risk_level"] == "critical").sum()) if "risk_level" in features_df.columns else None,
    "isolation_forest_anomalies": int((features_df["is_anomaly"] == 1).sum()) if "is_anomaly" in features_df.columns else None,
    "autoencoder_anomalies": int((features_df["autoencoder_is_anomaly"] == 1).sum()) if "autoencoder_is_anomaly" in features_df.columns else None,
    "alerts_with_email_activity": int((alerts_df["email_events"] > 0).sum()) if "email_events" in alerts_df.columns else None,
    "alerts_with_http_activity": int((alerts_df["http_events"] > 0).sum()) if "http_events" in alerts_df.columns else None,
    "final_number_of_columns": features_df.shape[1],
}

summary_df = pd.DataFrame(list(summary.items()), columns=["Metric", "Value"])

display(summary_df)

## 15. Conclusion

This notebook confirms that the UEBA pipeline generates exploitable analytical outputs.

It combines:

- exact summary tables;
- descriptive statistics;
- visual charts;
- contextual LDAP analysis;
- model anomaly summaries;
- high-risk alert samples.

The outputs can support the final analytical report, the Grafana dashboard interpretation, and future SOC-oriented investigation workflows.